# MATH 307 — Applied Linear Algebra
## Computational Exploration: LU and PLU Factorization

**Explore, Compute, Discover**

This notebook accompanies Section 1.1. It is designed for students with **no prior Python experience** as well as students who already know Python.

You do not need to write the factorization algorithms from scratch. The needed functions are provided. Early activities include starter code; you will mainly change matrix entries, run cells, make predictions, and explain what you observe.

Our pattern is:

**Theory → Prediction → Experiment → Observation → Explanation**


### A very small amount of Python

A matrix is entered with `np.array`. Each bracketed row below is one row of the matrix.

Run the cell using the **Run** button (or `Shift+Enter`). To experiment, change the numbers and run the cell again.


In [ ]:
import numpy as np

A = np.array([[1, 2],
              [3, 4]], dtype=float)

print(A)


### 1. LU factorization without row interchanges

The LU-Factorization Theorem in Section 1.1 tells us when an LU factorization can be constructed by Gaussian elimination without row interchanges.

The function below is the computational translation used in the textbook: elimination produces \(U\), and the opposites of the elimination multipliers are recorded in \(L\).

For now, simply **run this cell once** so that Python knows the function `lu_factor`.


In [ ]:
def lu_factor(A):
    A = np.array(A, dtype=float)
    m, n = A.shape
    U = A.copy()
    L = np.eye(m)

    for j in range(min(m, n)):
        if U[j, j] == 0:
            raise ValueError(
                "A row interchange is required."
            )

        for i in range(j + 1, m):
            multiplier = -U[i, j] / U[j, j]
            U[i, :] = U[i, :] + multiplier * U[j, :]
            L[i, j] = -multiplier

    return L, U


### 2. A first test

The matrix below is the one used in the initial LU construction in Section 1.1.

**Predict.** Before running the cell, compare it with your hand calculation. What \(L\) and \(U\) do you expect?

**Experiment.** Run the cell.

**Explain.** Compare the output with the factors obtained in the textbook.


In [ ]:
A = np.array([[ 1, -1,  3],
              [ 3, -1, 14],
              [-2,  4,  0]], dtype=float)

L, U = lu_factor(A)

print("L =")
print(L)
print("U =")
print(U)
print("A = LU:")
print(np.allclose(A, L @ U))


## Explore: From Theory to Computation

The following investigations correspond to the **Explore: From Theory to Computation** questions in Section 1.1.

For each investigation, try to **predict first**, then run or modify the code, and finally explain what the experiment tells you about the mathematics.


### Explore 1 — A rectangular matrix

Apply `lu_factor` to the \(3\times4\) matrix from Example 1.1.2.

**Predict.** What factors do you expect, especially for the unit-lower-triangular factorization?

**Experiment.** Starter code is provided below.

**Explain.** Verify \(A=LU\), compare with your hand factorization, and identify parts of the LU theorem's proof that you recognize in the algorithm.


In [ ]:
A = np.array([[ 2,  4,  6, -8],
              [ 3,  1, 19,  3],
              [-4, -1, -7,  4]], dtype=float)

L, U = lu_factor(A)

print("L =")
print(L)
print("U =")
print(U)
print("A = LU:")
print(np.allclose(A, L @ U))


In [ ]:
# Your notes / additional experiments for Explore 1:
# Change entries or enter another rectangular matrix below.


### Explore 2 — Make a row interchange necessary

Start with a matrix for which `lu_factor` works. Change **one entry** so that you predict a row interchange will become necessary.

**Predict.** Which entry will you change, and at which pivot do you expect the problem?

**Experiment.** Use the scaffold below. Replace the sample entries with your matrices.

**Explain.** Was your prediction correct? What changed in Gaussian elimination?


In [ ]:
# Matrix for which LU should work:
A1 = np.array([[1, 2, 0],
               [2, 5, 1],
               [1, 1, 3]], dtype=float)

# Change ONE entry to create your second matrix:
A2 = np.array([[1, 2, 0],
               [2, 5, 1],
               [1, 1, 3]], dtype=float)

# Try A1 first.
L, U = lu_factor(A1)
print("A1 = LU:", np.allclose(A1, L @ U))

# Then replace A1 by A2 below when you are ready.
# L, U = lu_factor(A2)


In [ ]:
# Use this empty cell for further matrices or observations.


### Explore 3 — Fail at the second pivot

Construct a \(3\times3\) matrix for which the **first pivot step succeeds** but `lu_factor` fails at the **second pivot**.

**Predict.** Write down your matrix and explain why you expect the second pivot to be zero after the first elimination step.

**Experiment.** Enter it below.

**Explain.** Use the hypothesis of the LU-Factorization Theorem to explain why the failure is mathematically meaningful rather than merely a programming error.


In [ ]:
# Enter your 3 x 3 matrix.
A = np.array([[..., ..., ...],
              [..., ..., ...],
              [..., ..., ...]], dtype=float)

# Remove the ... and replace them with numbers, then run:
# L, U = lu_factor(A)


In [ ]:
# Try another example if you wish.


### Explore 4 — A nonzero pivot can still be a poor numerical pivot

Start with a matrix that would require a row interchange because its first pivot is zero. Replace that zero by \(10^{-1}\), then \(10^{-3}\), and then \(10^{-6}\).

**Predict.** As the pivot becomes smaller, what do you expect to happen to some entries of \(L\) or \(U\)?

**Experiment.** The loop below performs the same experiment for all three pivot values.

**Explain.** What does this suggest about the difference between a pivot being merely nonzero and being a good pivot for numerical computation?


In [ ]:
for eps in [1e-1, 1e-3, 1e-6]:
    A = np.array([[eps, 1],
                  [1,   1]], dtype=float)

    L, U = lu_factor(A)

    print("pivot =", eps)
    print("L =")
    print(L)
    print("U =")
    print(U)
    print()


In [ ]:
# Explore other small nonzero pivots here.


### What did the LU experiments suggest?

Use this cell as a scratch space for additional computations. In your written work, summarize what your experiments suggest about:

- when LU without row interchanges succeeds;
- why a zero pivot matters;
- why a very small nonzero pivot can raise a different numerical issue.


In [ ]:
# Additional LU experiments:


## 3. Extending the algorithm: PLU factorization

The PLU-Factorization Theorem extends the theory by allowing row interchanges. The function below implements the constructive ideas used in its proof.

When a zero pivot occurs, the code searches below it for a nonzero entry and swaps rows. If a swap occurs after multipliers have already been stored in \(L\), the corresponding previously constructed entries of \(L\) must also move.

Run the next cell once to define `plu_factor`.


In [ ]:
def plu_factor(A):
    A = np.array(A, dtype=float)
    m, n = A.shape
    U = A.copy()
    L = np.eye(m)
    Q = np.eye(m)

    for j in range(min(m, n)):
        if U[j, j] == 0:
            pivot_row = None

            for k in range(j + 1, m):
                if U[k, j] != 0:
                    pivot_row = k
                    break

            if pivot_row is None:
                raise ValueError(
                    "No nonzero pivot is available in this column."
                )

            U[[j, pivot_row], :] = U[[pivot_row, j], :]
            Q[[j, pivot_row], :] = Q[[pivot_row, j], :]

            if j > 0:
                L[[j, pivot_row], :j] = L[[pivot_row, j], :j]

        for i in range(j + 1, m):
            multiplier = -U[i, j] / U[j, j]
            U[i, :] = U[i, :] + multiplier * U[j, :]
            L[i, j] = -multiplier

    P = Q.T
    return P, L, U


### A first PLU test

Use the matrix from Example 1.1.3.

**Predict.** Compare with the hand calculation before running the code.

**Experiment.** Run the cell.

**Explain.** Compare \(P,L,U\) with the factors obtained by hand and verify \(A=PLU\).


In [ ]:
A = np.array([[0, 0, 6],
              [1, 2, 3],
              [2, 1, 4]], dtype=float)

P, L, U = plu_factor(A)

print("P =")
print(P)
print("L =")
print(L)
print("U =")
print(U)
print("A = PLU:")
print(np.allclose(A, P @ L @ U))


## Explore: From Theorem to Algorithm

These investigations correspond to the PLU exploration in Section 1.1. Continue to use the pattern **Predict → Experiment → Explain**.


### PLU Explore 1 — Connect the proof and the code

For the matrix from Example 1.1.3, compare the computed \(P,L,U\) with the hand factors.

Identify in `plu_factor` the lines responsible for:

1. swapping rows of \(U\);
2. recording the row permutation;
3. moving previously stored entries of \(L\);
4. setting \(P=Q^T\).

Use the empty cell below if you want to print intermediate quantities or rerun the example.


In [ ]:
# Experiment with the Example 1.1.3 matrix here.


### PLU Explore 2 — Revisit your LU failure

Use the \(3\times3\) matrix you constructed in **LU Explore 3**, where `lu_factor` failed at the second pivot.

**Predict.** Will `plu_factor` now succeed?

**Experiment.** Re-enter your matrix below and verify \(A=PLU\).

**Explain.** Compare what the two algorithms do at the problematic pivot.


In [ ]:
# Re-enter your matrix from LU Explore 3.
A = np.array([[..., ..., ...],
              [..., ..., ...],
              [..., ..., ...]], dtype=float)

# Replace ... by numbers, then run:
# P, L, U = plu_factor(A)
# print("A = PLU:", np.allclose(A, P @ L @ U))


### PLU Explore 3 — Why must part of \(L\) be swapped?

Construct a matrix for which a row interchange occurs **after** at least one multiplier has already been stored in \(L\).

Temporarily add `print(L)` statements to `plu_factor`, or use the experimental copy below, to inspect \(L\) before and after the swap.

**Explain.** Why is this bookkeeping required to preserve the relation \(QA=LU\)?


In [ ]:
# Enter and investigate your matrix here.


### PLU Explore 4 — Form a conjecture

Experiment with several matrices of your own.

**Predict.** For each matrix, decide first whether you think ordinary LU will succeed or whether a row interchange will be needed.

**Experiment.** Test at least three matrices with `lu_factor` and `plu_factor`.

**Explain.** State a conjecture, based on your experiments, about when LU succeeds without row interchanges.


In [ ]:
# Matrix 1:

# Matrix 2:

# Matrix 3:


### PLU Explore 5 — Rectangular matrices and the limits of this code

Try `plu_factor` on a rectangular matrix.

The PLU-Factorization Theorem is more general than this particular implementation. This code stops if the current pivot column contains no available nonzero pivot, even in a situation where elimination could continue by moving to a later pivot column.

**Experiment.** Try rectangular examples and see whether you can produce this behavior.

**Explain.** Distinguish carefully between a limitation of the theorem and a limitation of this implementation.


In [ ]:
# Enter a rectangular matrix and experiment here.


## Theory and computational thinking

Theorems provide the mathematical foundation, but a theorem is not yet a program. An implementation requires additional decisions: how to search for pivots, how to store and update factors, what to do in exceptional cases, and how finite-precision arithmetic affects the computation.

As you progress through the textbook, the notebooks will gradually provide less starter code. You will increasingly make computational choices yourself. **Prior Python experience is not assumed**; the goal is to develop the computational thinking alongside the linear algebra.


---

## Exercise 38 — Recovering a blurred image using Python

A grayscale image is represented by a $20\times20$ matrix $X$, where $0$ represents black and $1$ represents white. The starter code creates an image of the letter H.
Suppose the image is blurred vertically according to
$$
B=AX,\qquad A=\frac14
\begin{bmatrix}
3&1&0&\cdots&0\\
1&2&1&\ddots&\vdots\\
0&1&2&\ddots&0\\
\vdots&\ddots&\ddots&\ddots&1\\
0&\cdots&0&1&3
\end{bmatrix}_{20\times20}.
$$
All entries outside the three indicated diagonals are zero. To recover the image, we solve $AX=B$, one column at a time.

**Solve all parts using Python.** Run the setup cell, then complete (a)–(d) in order. Worked examples are followed by empty code cells for your tasks and empty text cells for your explanations.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def lu_factor(A):
    A = np.array(A, dtype=float)
    m, n = A.shape
    U = A.copy()
    L = np.eye(m)

    for j in range(min(m, n)):
        if U[j, j] == 0:
            raise ValueError(
                "A row interchange is required."
            )

        for i in range(j + 1, m):
            multiplier = -U[i, j] / U[j, j]
            U[i, :] = U[i, :] + multiplier * U[j, :]
            L[i, j] = -multiplier

    return L, U

# Solve LY = B and UZ = Y for every column of B.
def lu_solve(L, U, B):
    n = L.shape[0]
    Y = np.zeros_like(B, dtype=float)
    Z = np.zeros_like(B, dtype=float)
    for i in range(n):
        Y[i] = B[i] - L[i, :i] @ Y[:i]
    for i in range(n - 1, -1, -1):
        Z[i] = (Y[i] - U[i, i+1:] @ Z[i+1:]) / U[i, i]
    return Z

def show_images(images, titles):
    fig, axes = plt.subplots(1, len(images), figsize=(4*len(images), 4))
    for ax, image, title in zip(axes, images, titles):
        ax.imshow(image, cmap="gray", vmin=0, vmax=1)
        ax.set_title(title)
        ax.axis("off")
    plt.tight_layout()
    plt.show()


### (a)

Run the supplied code to display the original H and its blurred image. Which features are most affected by the blur? Then modify the image matrix $X$, for example by moving the horizontal bar or making it thicker. Recompute $B=AX$ and display your modified image and its blur. Use this modified image throughout the remaining parts.

In [ ]:
X = np.zeros((20, 20))
X[3:17, 4:7] = 1
X[3:17, 13:16] = 1
X[9:12, 4:16] = 1

A = (2*np.eye(20) + np.eye(20, k=1) + np.eye(20, k=-1))/4
A[0, 0] = A[-1, -1] = 3/4
B = A @ X

show_images([X, B], ["Original", "Blurred"])


**Your code:** Copy the image-creation commands into the empty cell below and change the row ranges that define the H. For example, `X[9:12, 4:16] = 1` fills rows 9–11 and columns 4–15 with white pixels (indices start at 0; the upper endpoint is excluded). Recreate `X` from zeros before drawing your new image, so old strokes do not remain. Then recompute `B = A @ X` and use `show_images` as in the example. Keep `A` unchanged.


**Your explanation:** Describe how the blur affects the original and modified images in the empty text cell below. Double-click a text cell to edit it; press Shift+Enter to finish.

### (b)

Use the supplied Python function `lu_factor` to compute an LU factorization of the blur matrix $A$. Store the factors as `L` and `U`, and verify that $A=LU$ using `np.allclose(A, L @ U)`. Then adapt the supplied single-column recovery example to recover all $20$ columns of the image. Display the recovered image $\widehat X$ and calculate the largest absolute entry of $\widehat X-X$. Explain why the same factors can be used for every column.

**Example with a different matrix.** Run this cell to see how to apply `lu_factor` and verify its output. The example uses `L_C` and `U_C` so the factors for your blur matrix remain separate.


In [ ]:
C = np.array([[2, 1],
              [4, 3]], dtype=float)
L_C, U_C = lu_factor(C)
print("L_C =", L_C, sep="\n")
print("U_C =", U_C, sep="\n")
print("C = L_C @ U_C:", np.allclose(C, L_C @ U_C))


**Your code:** Apply `lu_factor` to `A`, store the results as `L` and `U`, and check their product using `np.allclose`.

**Example of solving for one column.** The function `lu_solve` solves first with the lower triangular factor, then with the upper triangular factor. It accepts either one column or several columns at once.


In [ ]:
d = np.array([[1], [2]], dtype=float)
z = lu_solve(L_C, U_C, d)
print("Solution for one column:", z, sep="\n")
print("Largest equation error:", np.max(np.abs(C @ z - d)))


**Your code:** Adapt the example by supplying `L`, `U`, and the entire matrix `B`. Store the recovered image as `X_recovered`. Use `show_images` to display it beside `X`. The expression `np.max(np.abs(P - Q))` gives the largest absolute difference between two arrays `P` and `Q`; adapt it to calculate your recovery error.

**Your explanation:** Explain why one LU factorization suffices for all 20 columns, and interpret your recovery error.

### (c)

Run the supplied code to add noise:
$$
B_\varepsilon=B+\varepsilon E,\qquad \varepsilon=0.005,
$$
where the entries of $E$ are randomly chosen between $-1$ and $1$. Write code to recover an image $\widehat X$ from $B_\varepsilon$, reusing your existing factors $L$ and $U$. Display the recovered image and calculate its largest absolute difference from $X$. Describe the effect of the noise.

Run this supplied noise-generation cell once. It uses your modified image from part (a).

In [ ]:
rng = np.random.default_rng(1)
E = rng.uniform(-1, 1, B.shape)
epsilon = 0.005
B_noisy = B + epsilon*E


**Your code:** Adapt your recovery commands from part (b) to use `B_noisy`. Store the result as `X_noisy`. Display it beside `X` and `B_noisy`, and calculate the largest recovery error. Reuse `L` and `U`; do not factor `A` again.

**Your explanation:** Describe the effect of noise on the recovered image.

### (d)

Adapt your code to repeat part (c) with $\varepsilon=0.001$ and $0.01$, keeping the same $E$. For each noise level, calculate the largest absolute entries of
$$
\widehat X-X\qquad\text{and}\qquad A\widehat X-B_\varepsilon.
$$
How does the recovery error change with the noise level? Does a small equation error guarantee that the recovered image is close to the original? Support your answers with your results.

**Your code:** Copy and adapt your commands from part (c). Set `epsilon = 0.001`, recompute `B_noisy` using the existing `E`, and recover the image using the existing `L` and `U`. Calculate the recovery error as before. Adapt the equation-error example in part (b) to measure the largest absolute entry of `A @ X_noisy - B_noisy`. Then repeat with `epsilon = 0.01`. You may use separate cells or change the value and rerun. Do not regenerate `E`.

**Your explanation:** Record both errors for each noise level, then answer the two questions in part (d).

**Display note.** Every image uses the same grayscale range, from 0 to 1. Recovered values outside this range appear black or white, but their numerical values are retained when measuring errors.
